# Router Stage 1 v3 — Gatekeeper Fine-tune (resolution-degradation fix)

**Goal:** fix the confirmed root cause of the breast_histology rejection bug — the gatekeeper
learned "blocky low-res texture -> medical" from breast_histology's 50x50px patches (upscaled
4.5x) and rejects crisp, full-resolution real-world histology images.

**Fix:** add a random downsample->upscale augmentation across ALL gatekeeper training images
(both classes, all sources), then fine-tune from the existing `router_stage1_v2` checkpoint —
few epochs, low LR — rather than a full retrain, since the model is already correct
(0.9996 overall, per-source >=0.9989) on everything else.

**Before running:** Steps 1-8 below (config / negative+positive source loading / train-val-test-
generalization split / `label_to_idx`) are UNCHANGED from `router_stage1_v2_training_kaggle.ipynb`
— paste those cells in at the marked spot. Nothing about the data sources or split logic changes
for this fine-tune; only the train-time transform and the starting checkpoint change.

Conventions followed (same as brain_mri_v2 retrain): `num_workers=4` fixed, non-deprecated
`torch.amp` calls, `channels_last`, per-epoch checkpointing to a `.checkpoint_inprogress.pt`
file on val_loss improvement with a resume cell — reliability over peak throughput.

## Step 0 — Imports & config

In [2]:
import sys
import random
from pathlib import Path
import shutil
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

PROJECT_ROOT = Path(r"E:\programming\CliniqAI")
MODELS_ROOT = PROJECT_ROOT / "models"
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.inference.model_loader import load_efficientnet_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(device)

cuda


## Steps 1-8 — PASTE FROM `router_stage1_v2_training_kaggle.ipynb`

Copy the cells that build (unchanged from v2):
- the 6 Kaggle negative sources + synthetic screenshots + positive-class specialist sampling
- `train_records`, `val_records`, `test_records`, `generalization_records`
- `label_to_idx`

Nothing here needs to change for the fine-tune — same data, same split. Paste those cells into
this notebook right below this one, in place of this markdown cell, then continue to Step 9
below.

In [3]:
PROJECT_ROOT = Path(r"E:\\programming\\CliniqAI")
BACKEND_ROOT = PROJECT_ROOT / "backend"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
NEG_ROOT = PROJECT_ROOT / "data" / "raw" / "gatekeeper_negatives_v2"
NEG_ROOT.mkdir(parents=True, exist_ok=True)

# Existing specialist folders -- source of the POSITIVE (medical) class.
# We only care about "is this a medical image", so we glob every image file
# under each folder regardless of internal class-subfolder structure.
POSITIVE_SOURCES = {
    "brain_mri": DATA_RAW / "brain_tumor",
    "kidney_ct": DATA_RAW / "kidney_stone",
    "retinal": DATA_RAW / "diabetic_retinopathy",
    "skin_lesion": DATA_RAW / "skin_lesion",
    "breast_histology": DATA_RAW / "breast_histology",
    "bone_fracture": DATA_RAW / "bone_fracture",
    "pneumonia": DATA_RAW / "pneumonia",
}

# Kaggle dataset slug -> local subfolder name. All verified to exist on Kaggle at time of writing.
KAGGLE_NEGATIVE_SOURCES = {
    "food101":        ("kmader/food41",                    "kaggle_food101"),        # ~101,000 food images
    "scenes":         ("puneet6060/intel-image-classification", "kaggle_scenes"),    # ~25,000 natural scenes (buildings/forest/glacier/mountain/sea/street)
    "natural_images": ("prasunroy/natural-images",          "kaggle_natural_images"), # ~6,900 images, 8 object classes
    "imagenet_mini":  ("ifigotin/imagenetmini-1000",        "kaggle_imagenet_mini"),  # ~34,000 images, 1000 ImageNet classes -- broadest diversity here
    "fruits":         ("moltean/fruits",                    "kaggle_fruits"),         # ~90,000+ images, fruits/vegetables -- directly relevant (apple was a failure case)
    "human_faces":    ("ashwingupta3012/human-faces",       "kaggle_human_faces"),    # web-scraped human face photos -- selfie/person coverage
}

IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Target images per positive modality and per negative source -- caps at whatever
# is actually available, so this is a ceiling, not a guarantee.
TARGET_PER_POSITIVE_SOURCE = 12000
TARGET_PER_NEGATIVE_SOURCE = 12000
NUM_SYNTHETIC_SCREENSHOTS = 6000

BATCH_SIZE = 64          # safe with AMP on 6GB VRAM (v1 without AMP overflowed at 64 -- AMP roughly halves activation memory)
NUM_WORKERS = 4
EPOCHS = 30
PATIENCE = 7
LR = 1e-3
WEIGHT_DECAY = 1e-4

CHECKPOINT_DIR = PROJECT_ROOT / "models" / "router"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "router_stage1_v2_efficientnet_b0.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [4]:
usage = shutil.disk_usage(PROJECT_ROOT.drive + "\\\\")
free_gb = usage.free / 1024**3
estimated_need_gb = 15

print(f"Free space on {PROJECT_ROOT.drive}: {free_gb:.1f} GB")
print(f"Estimated space needed for this notebook's downloads: ~{estimated_need_gb} GB")
if free_gb < estimated_need_gb + 5:
    print("\n[WARNING] Free space is close to the estimated requirement. Consider freeing space "
          "or dropping one source from KAGGLE_NEGATIVE_SOURCES before continuing.")
else:
    print("\nLooks fine, continuing.")

Free space on E:: 105.6 GB
Estimated space needed for this notebook's downloads: ~15 GB

Looks fine, continuing.


In [5]:
def kaggle_download(slug, dest_folder):
    dest = NEG_ROOT / dest_folder
    dest.mkdir(parents=True, exist_ok=True)
    existing = list(dest.rglob("*"))
    if len(existing) > 5:
        print(f"[skip] {slug} -- {dest_folder} already has {len(existing)} files.")
        return
    print(f"Downloading {slug} -> {dest_folder} ...")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"[ERROR] {slug} failed:\n{result.stderr}")
        print("Check that 'kaggle' is on PATH (pip install kaggle) and kaggle.json is valid.")
    else:
        n_files = len(list(dest.rglob("*")))
        print(f"  Done -- {n_files} files extracted.")

for tag, (slug, folder) in KAGGLE_NEGATIVE_SOURCES.items():
    kaggle_download(slug, folder)

[skip] kmader/food41 -- kaggle_food101 already has 101121 files.
[skip] puneet6060/intel-image-classification -- kaggle_scenes already has 24353 files.
[skip] prasunroy/natural-images -- kaggle_natural_images already has 13817 files.
[skip] ifigotin/imagenetmini-1000 -- kaggle_imagenet_mini already has 40671 files.
[skip] moltean/fruits -- kaggle_fruits already has 337020 files.
[skip] ashwingupta3012/human-faces -- kaggle_human_faces already has 7220 files.


In [6]:
from PIL import Image, ImageDraw
import random as _rnd

synthetic_root = NEG_ROOT / "synthetic_screenshots"
synthetic_root.mkdir(exist_ok=True)

LOREM_WORDS = ("lorem ipsum dolor sit amet consectetur adipiscing elit sed do eiusmod tempor "
               "incididunt ut labore et dolore magna aliqua enim minim veniam quis nostrud "
               "exercitation ullamco laboris nisi aliquip ex ea commodo consequat duis aute "
               "irure reprehenderit voluptate velit esse cillum eu fugiat nulla pariatur "
               "excepteur sint occaecat cupidatat non proident sunt culpa qui officia deserunt "
               "mollit anim id est laborum").split()

def random_text(n_words):
    return " ".join(_rnd.choice(LOREM_WORDS) for _ in range(n_words))

def make_synthetic_image(size=(600, 800)):
    kind = _rnd.choice(["document", "chat_screenshot", "code_editor", "spreadsheet", "webpage"])
    bg = _rnd.choice(["#ffffff", "#f5f5f5", "#1e1e1e", "#0d1117", "#eef1f5"])
    img = Image.new("RGB", size, bg)
    draw = ImageDraw.Draw(img)
    dark_bg = bg in ("#1e1e1e", "#0d1117")
    text_color = "#e6e6e6" if dark_bg else "#222222"
    accent = _rnd.choice(["#4285f4", "#25d366", "#0078d4", "#ff5722", "#6c5ce7"])

    if kind in ("document", "spreadsheet"):
        y = 40
        while y < size[1] - 40:
            line_words = _rnd.randint(4, 12)
            draw.text((40, y), random_text(line_words), fill=text_color)
            y += _rnd.randint(22, 34)
        if kind == "spreadsheet":
            for gx in range(0, size[0], _rnd.randint(60, 120)):
                draw.line([(gx, 0), (gx, size[1])], fill="#cccccc", width=1)
            for gy in range(0, size[1], _rnd.randint(30, 50)):
                draw.line([(0, gy), (size[0], gy)], fill="#cccccc", width=1)
    elif kind == "chat_screenshot":
        y = 30
        for _ in range(_rnd.randint(6, 14)):
            bubble_w = _rnd.randint(150, 400)
            bubble_h = _rnd.randint(30, 60)
            x = _rnd.choice([30, size[0] - bubble_w - 30])
            color = accent if x == 30 else "#e0e0e0"
            draw.rounded_rectangle([x, y, x + bubble_w, y + bubble_h], radius=12, fill=color)
            y += bubble_h + 15
    elif kind == "code_editor":
        img = Image.new("RGB", size, "#1e1e1e")
        draw = ImageDraw.Draw(img)
        y = 20
        indent_levels = [0, 20, 40, 20, 0]
        for i in range(_rnd.randint(15, 30)):
            indent = _rnd.choice(indent_levels)
            width = _rnd.randint(80, 350)
            color = _rnd.choice(["#569cd6", "#9cdcfe", "#ce9178", "#4ec9b0", "#d4d4d4"])
            draw.rectangle([40 + indent, y, 40 + indent + width, y + 12], fill=color)
            y += 20
    else:  # webpage
        draw.rectangle([0, 0, size[0], 70], fill=accent)
        y = 100
        while y < size[1] - 40:
            draw.rectangle([40, y, size[0] - 40, y + 12], fill="#cccccc" if not dark_bg else "#444444")
            y += 26

    return img

n_existing = len(list(synthetic_root.glob("*.png")))
n_to_make = max(0, NUM_SYNTHETIC_SCREENSHOTS - n_existing)
print(f"Generating {n_to_make} synthetic screenshot/document images...")
for i in range(n_to_make):
    img = make_synthetic_image()
    img.save(synthetic_root / f"synthetic_{i:05d}.png")
    if (i + 1) % 1000 == 0:
        print(f"  {i+1}/{n_to_make}")
print("Done.")

Generating 0 synthetic screenshot/document images...
Done.


In [7]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_images(root, limit=None):
    if not root.exists():
        return []
    return [p for p in root.rglob("*") if p.suffix.lower() in IMG_EXTENSIONS]

positive_records = []  # (path, source_tag)
print("Positive (medical) sources:")
for tag, folder in POSITIVE_SOURCES.items():
    all_imgs = find_images(folder)
    if not all_imgs:
        print(f"  [WARNING] {tag}: no images found under {folder} -- skipping. "
              f"Check the folder path if this is unexpected.")
        continue
    n = min(len(all_imgs), TARGET_PER_POSITIVE_SOURCE)
    sampled = random.sample(all_imgs, n)
    positive_records.extend((p, tag) for p in sampled)
    print(f"  {tag:<20} available={len(all_imgs):<7} sampled={n}")

print(f"\nTotal positive images: {len(positive_records)}")

Positive (medical) sources:
  brain_mri            available=7200    sampled=7200
  kidney_ct            available=12446   sampled=12000
  retinal              available=3662    sampled=3662
  skin_lesion          available=25331   sampled=12000
  breast_histology     available=555048  sampled=12000
  bone_fracture        available=10581   sampled=10581
  pneumonia            available=4751    sampled=4751

Total positive images: 62194


In [8]:
negative_records = []  # (path, source_tag)
print("Negative (non-medical) sources:")

for tag, (slug, folder) in KAGGLE_NEGATIVE_SOURCES.items():
    all_imgs = find_images(NEG_ROOT / folder)
    if not all_imgs:
        print(f"  [WARNING] {tag}: no images found under {NEG_ROOT / folder} -- did the Kaggle "
              f"download succeed? Re-run Step 3's cell if not.")
        continue
    n = min(len(all_imgs), TARGET_PER_NEGATIVE_SOURCE)
    sampled = random.sample(all_imgs, n)
    negative_records.extend((p, tag) for p in sampled)
    print(f"  {tag:<20} available={len(all_imgs):<8} sampled={n}")

synth_imgs = find_images(synthetic_root)
n = min(len(synth_imgs), TARGET_PER_NEGATIVE_SOURCE)
sampled = random.sample(synth_imgs, n)
negative_records.extend((p, "synthetic_screenshots") for p in sampled)
print(f"  {'synthetic_screenshots':<20} available={len(synth_imgs):<8} sampled={n}")

print(f"\nTotal negative images: {len(negative_records)}")
print(f"Total positive images: {len(positive_records)}")
print("(A large imbalance here is fine -- same-order-of-magnitude match is what matters.)")

Negative (non-medical) sources:
  food101              available=101000   sampled=12000
  scenes               available=24335    sampled=12000
  natural_images       available=13798    sampled=12000
  imagenet_mini        available=38668    sampled=12000
  fruits               available=335456   sampled=12000
  human_faces          available=7219     sampled=7219
  synthetic_screenshots available=6000     sampled=6000

Total negative images: 73219
Total positive images: 62194
(A large imbalance here is fine -- same-order-of-magnitude match is what matters.)


In [9]:
HELD_OUT_SOURCE = "fruits"  # never included in train/val -- generalization-only

generalization_records = [(p, "negative", tag) for p, tag in negative_records if tag == HELD_OUT_SOURCE]
trainable_negative = [(p, "negative", tag) for p, tag in negative_records if tag != HELD_OUT_SOURCE]
trainable_positive = [(p, "medical", tag) for p, tag in positive_records]

all_trainable = trainable_positive + trainable_negative
random.shuffle(all_trainable)

def stratified_split(records, val_frac=0.15, test_frac=0.15):
    buckets = defaultdict(list)
    for rec in records:
        buckets[(rec[1], rec[2])].append(rec)
    train, val, test = [], [], []
    for key, items in buckets.items():
        random.shuffle(items)
        n = len(items)
        n_val = int(n * val_frac)
        n_test = int(n * test_frac)
        val.extend(items[:n_val])
        test.extend(items[n_val:n_val + n_test])
        train.extend(items[n_val + n_test:])
    random.shuffle(train)
    return train, val, test

train_records, val_records, test_records = stratified_split(all_trainable)

print(f"Train: {len(train_records)}")
print(f"Val:   {len(val_records)}")
print(f"Test:  {len(test_records)}  (in-distribution, held-out split of trained sources)")
print(f"Generalization set: {len(generalization_records)}  (from '{HELD_OUT_SOURCE}', NEVER trained on)")

CLASS_NAMES = ["non_medical", "medical"]
label_to_idx = {"negative": 0, "medical": 1}

Train: 86393
Val:   18510
Test:  18510  (in-distribution, held-out split of trained sources)
Generalization set: 12000  (from 'fruits', NEVER trained on)


total images=135413

In [10]:
%%writefile E:\\programming\\CliniqAI\\scripts\\gatekeeper_dataset_v2.py
"""
Standalone, importable Dataset for Router Stage 1 v2 training.
Must live in its own module (not a notebook cell) for Windows DataLoader
multiprocessing (num_workers > 0) to work -- worker subprocesses re-import
this module fresh, so ImageFile.LOAD_TRUNCATED_IMAGES must be set here at
module level, not just in the training notebook.
"""
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class GatekeeperDatasetV2(Dataset):
    """records: list of (path, label_str, source_tag). label_str in {'medical','negative'}."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _tag = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            # Runtime fallback: corrupted/unreadable file -- substitute a random
            # different sample instead of crashing the whole training run.
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label


Overwriting E:\\programming\\CliniqAI\\scripts\\gatekeeper_dataset_v2.py


## Step 9 — Dataset module update: add the resolution-degradation augmentation

Rewrites `scripts/gatekeeper_dataset_v2.py` with the original `GatekeeperDatasetV2` class
(unchanged — reproduced exactly from the existing file, confirmed via `nbconvert` this session,
not guessed) plus the new `RandomResolutionDegradation` transform. Standalone module file, not an
inline notebook class, so Windows `num_workers>0` multiprocessing still works.

In [11]:
%%writefile {PROJECT_ROOT}/scripts/gatekeeper_dataset_v2.py
"""
Standalone, importable Dataset for Router Stage 1 v2/v3 training.
Must live in its own module (not a notebook cell) for Windows DataLoader
multiprocessing (num_workers > 0) to work -- worker subprocesses re-import
this module fresh, so ImageFile.LOAD_TRUNCATED_IMAGES must be set here at
module level, not just in the training notebook.
"""
import random
from pathlib import Path
from PIL import Image, ImageFile
from torch.utils.data import Dataset

ImageFile.LOAD_TRUNCATED_IMAGES = True


class GatekeeperDatasetV2(Dataset):
    """records: list of (path, label_str, source_tag). label_str in {'medical','negative'}."""

    def __init__(self, records, label_to_idx, transform=None):
        self.records = records
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def _load(self, idx):
        path, label_str, _tag = self.records[idx]
        img = Image.open(path).convert("RGB")
        img.load()
        label = self.label_to_idx[label_str]
        return img, label

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
        except (OSError, Image.DecompressionBombError):
            # Runtime fallback: corrupted/unreadable file -- substitute a random
            # different sample instead of crashing the whole training run.
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
        if self.transform:
            img = self.transform(img)
        return img, label


class RandomResolutionDegradation:
    """Randomly downsamples then upscales back, breaking the 'blocky texture -> medical'
    shortcut the v2 gatekeeper learned from breast_histology's 50px patches upscaled 4.5x.
    Applied only in the train transform -- eval/test/generalization stay untouched so existing
    per-source accuracy numbers remain comparable."""

    def __init__(self, p=0.5, low_res_range=(32, 96)):
        self.p = p
        self.low_res_range = low_res_range

    def __call__(self, img):
        if random.random() > self.p:
            return img
        w, h = img.size
        low = random.randint(*self.low_res_range)
        img = img.resize((low, low), Image.BILINEAR)
        img = img.resize((w, h), Image.BILINEAR)
        return img

Overwriting E:\programming\CliniqAI/scripts/gatekeeper_dataset_v2.py


## Step 10 — Transforms & DataLoaders

In [12]:
from scripts.gatekeeper_dataset_v2 import GatekeeperDatasetV2, RandomResolutionDegradation

train_transform = transforms.Compose([
    RandomResolutionDegradation(p=0.5, low_res_range=(32, 96)),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = GatekeeperDatasetV2(train_records, label_to_idx, transform=train_transform)
val_ds = GatekeeperDatasetV2(val_records, label_to_idx, transform=eval_transform)
test_ds = GatekeeperDatasetV2(test_records, label_to_idx, transform=eval_transform)
generalization_ds = GatekeeperDatasetV2(generalization_records, label_to_idx, transform=eval_transform)

train_loader = DataLoader(
    train_ds, batch_size=64, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
test_loader = DataLoader(
    test_ds, batch_size=64, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)
generalization_loader = DataLoader(
    generalization_ds, batch_size=64, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4,
)

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} generalization={len(generalization_ds)}")

train=86393 val=18510 test=18510 generalization=12000


## Step 11 — Load the v2 checkpoint to fine-tune from

In [13]:
CKPT_IN = MODELS_ROOT / "router" / "router_stage1_v2_efficientnet_b0.pt"
CKPT_OUT = MODELS_ROOT / "router" / "router_stage1_v3_efficientnet_b0.pt"
CKPT_INPROGRESS = MODELS_ROOT / "router" / "router_stage1_v3.checkpoint_inprogress.pt"

model, class_names = load_efficientnet_checkpoint(CKPT_IN, device, model_name="router_stage1")
model = model.to(device, memory_format=torch.channels_last)
print("loaded", CKPT_IN.name, "classes:", class_names)

[loaded] router_stage1: 2 classes -> ['non_medical', 'medical']
loaded router_stage1_v2_efficientnet_b0.pt classes: ['non_medical', 'medical']


## Step 12 — Resume from an in-progress fine-tune, if one exists

Run this cell every time before training — it's a no-op on a fresh start, and picks up from the last val_loss-improved epoch if this was interrupted.

In [14]:
start_epoch = 0
best_val_loss = float("inf")

if CKPT_INPROGRESS.exists():
    ckpt = torch.load(CKPT_INPROGRESS, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print(f"resumed from epoch {ckpt['epoch']} (best_val_loss={best_val_loss:.4f})")
else:
    print("starting fresh")

resumed from epoch 2 (best_val_loss=0.0006)


C:\Users\anish\AppData\Local\Temp\ipykernel_9572\4292197965.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_INPROGRESS, map_location=device)


## Step 13 — Fine-tune loop

Low LR, few epochs — this is a fine-tune, not a full retrain, since v2 is already correct
everywhere except the one shortcut this augmentation targets.

In [15]:
N_EPOCHS = 5  # fine-tune only

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)
scaler = torch.amp.GradScaler(device="cuda", enabled=(device.type == "cuda"))


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, memory_format=torch.channels_last, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                out = model(imgs)
                loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total


for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, memory_format=torch.channels_last, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_ds)
    val_loss, val_acc = evaluate(model, val_loader)
    scheduler.step(val_loss)

    print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "classes": class_names,
            "epoch": epoch,
            "best_val_loss": best_val_loss,
        }, CKPT_INPROGRESS)
        print(f"  -> checkpoint saved (val_loss improved to {best_val_loss:.4f})")

print("done")

epoch 3: train_loss=0.0067 val_loss=0.0007 val_acc=0.9997
epoch 4: train_loss=0.0064 val_loss=0.0006 val_acc=0.9997
done


## Step 14 — Per-source evaluation (v3 vs v2 comparison)

Same per-source breakdown format used for v2, plus the held-out `fruits` generalization set.
Fill in `records_by_source` the same way the v2 notebook does (grouping test_records/
generalization_records by their `source_tag`), then run each source through `evaluate()` with
a per-source loader.

In [ ]:
class TaggedGatekeeperDataset(GatekeeperDatasetV2):
    """Same as GatekeeperDatasetV2 but also returns the source_tag per sample,
    so per-source accuracy can be computed in a single pass instead of one
    DataLoader per source (avoids repeated Windows worker-spawn overhead)."""

    def __getitem__(self, idx):
        try:
            img, label = self._load(idx)
            used_idx = idx
        except (OSError, Image.DecompressionBombError):
            fallback_idx = (idx + 1) % len(self.records)
            img, label = self._load(fallback_idx)
            used_idx = fallback_idx
        if self.transform:
            img = self.transform(img)
        source_tag = self.records[used_idx][2]
        return img, label, source_tag


def evaluate_per_source(model, records, label_to_idx, transform, batch_size=64):
    ds = TaggedGatekeeperDataset(records, label_to_idx, transform=transform)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True, persistent_workers=True, prefetch_factor=4,
    )
    correct = defaultdict(int)
    total = defaultdict(int)
    model.eval()
    with torch.no_grad():
        for imgs, labels, sources in loader:
            imgs = imgs.to(device, memory_format=torch.channels_last, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                out = model(imgs)
            preds = out.argmax(1).cpu()
            for p, l, s in zip(preds, labels.cpu(), sources):
                total[s] += 1
                correct[s] += int(p == l)
    return {s: correct[s] / total[s] for s in total}


print("=== v3 test set, per source ===")
for source, acc in sorted(evaluate_per_source(model, test_records, label_to_idx, eval_transform).items()):
    print(f"{source:25s} {acc:.4f}")

print("\n=== v3 generalization set (held-out source), per source ===")
for source, acc in sorted(evaluate_per_source(model, generalization_records, label_to_idx, eval_transform).items()):
    print(f"{source:25s} {acc:.4f}")

=== v3 test set, per source ===


## Step 15 — Real-world sanity check: the Wikimedia breast_histology image

Reproduces isolated test #3 from the handoff doc against the NEW v3 checkpoint. This is the
actual bug-fix confirmation — if this comes back `medical`, the fix worked.

In [ ]:
from PIL import Image as PILImage

SANITY_IMG = PROJECT_ROOT / "data" / "raw" / "manual_sanity_check" / "Normal_breast_histology.png"

img = PILImage.open(SANITY_IMG).convert("RGB")
x = eval_transform(img).unsqueeze(0).to(device, memory_format=torch.channels_last)

model.eval()
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
        logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]

pred_idx = probs.argmax().item()
print(f"prediction: {class_names[pred_idx]}  (confidence {probs[pred_idx].item():.4f})")
for i, c in enumerate(class_names):
    print(f"  {c}: {probs[i].item():.4f}")

## Step 16 — Save final checkpoint

Only run once you're satisfied with the per-source numbers above and the sanity check passes.
Copy `router_stage1_v3_efficientnet_b0.pt` into `backend/models/router/` and update
`backend/app/config.py`'s `ROUTER_STAGE1_PATH` manually — no automatic sync between
`models/` and `backend/models/` yet (per the handoff doc).

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "classes": class_names,
    "num_classes": len(class_names),
    "architecture": "efficientnet_b0_wide512",
}, CKPT_OUT)
print("saved:", CKPT_OUT)